In [1]:
import sqlite3

In [2]:
conn = sqlite3.connect("water_stress.db")
cursor = conn.cursor()

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Districts (
    District_ID INTEGER PRIMARY KEY,
    Name TEXT,
    Total_Area_SQKM REAL,
    Population_2023 INTEGER,
    Industrial_Zone_Count INTEGER,
    Lat REAL,
    Long REAL
)
""")

In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Date (
    Date TEXT PRIMARY KEY,
    Year INTEGER,
    Month INTEGER,
    Quarter INTEGER,
    Season TEXT
)
""")

In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Fact_Rainfall (
    Record_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    District_ID INTEGER,
    Date TEXT,
    Precipitation_mm REAL,
    Historical_Avg_mm REAL,
    FOREIGN KEY (District_ID) REFERENCES Dim_Districts(District_ID),
    FOREIGN KEY (Date) REFERENCES Dim_Date(Date)
)
""")

In [6]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Fact_Groundwater (
    Measurement_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    District_ID INTEGER,
    Date TEXT,
    Water_Table_Depth_m REAL,
    Recharge_Rate REAL,
    FOREIGN KEY (District_ID) REFERENCES Dim_Districts(District_ID),
    FOREIGN KEY (Date) REFERENCES Dim_Date(Date)
)
""")

In [7]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Fact_Reservoirs (
    Res_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    District_ID INTEGER,
    Date TEXT,
    Current_Capacity_TMC REAL,
    Max_Capacity_TMC REAL,
    FOREIGN KEY (District_ID) REFERENCES Dim_Districts(District_ID),
    FOREIGN KEY (Date) REFERENCES Dim_Date(Date)
)
""")

In [8]:
conn.commit()
conn.close()
print(" Database and tables created successfully!")

 Database and tables created successfully!


In [9]:
import pandas as pd
import numpy as np
import sqlite3
import random
from datetime import datetime, timedelta

# =========================================================
# CONFIG
# =========================================================
np.random.seed(42)
random.seed(42)

db_name = "telangana_water_stress_star_schema.db"

# =========================================================
# 1. DIM_DISTRICTS TABLE
# =========================================================
district_names = [
    "Adilabad", "Bhadradri Kothagudem", "Hanamkonda", "Hyderabad",
    "Jagtial", "Jangaon", "Jayashankar Bhupalpally", "Jogulamba Gadwal",
    "Kamareddy", "Karimnagar", "Khammam", "Komaram Bheem Asifabad",
    "Mahabubabad", "Mahabubnagar", "Mancherial", "Medak", "Medchal-Malkajgiri",
    "Mulugu", "Nagarkurnool", "Nalgonda", "Narayanpet", "Nirmal",
    "Nizamabad", "Peddapalli", "Rajanna Sircilla", "Ranga Reddy",
    "Sangareddy", "Siddipet", "Suryapet", "Vikarabad", "Wanaparthy",
    "Warangal", "Yadadri Bhuvanagiri"
]

# Approximate lat/long ranges for Telangana
lat_range = (15.8, 19.9)
long_range = (77.2, 81.0)

district_rows = []
for i, name in enumerate(district_names, start=1):
    total_area = round(np.random.uniform(1800, 4500), 2)
    population = int(np.random.uniform(300000, 4500000))
    industrial_zones = int(np.random.randint(1, 15))
    lat = round(np.random.uniform(*lat_range), 6)
    lon = round(np.random.uniform(*long_range), 6)

    district_rows.append([
        i, name, total_area, population, industrial_zones, lat, lon
    ])

dim_districts = pd.DataFrame(
    district_rows,
    columns=[
        "District_ID", "Name", "Total_Area_SQKM", "Population_2023",
        "Industrial_Zone_Count", "Lat", "Long"
    ]
)

# =========================================================
# 2. DIM_DATE TABLE
# =========================================================
start_date = datetime(2023, 1, 1)
num_days = 365

date_rows = []
for i in range(num_days):
    current_date = start_date + timedelta(days=i)
    year = current_date.year
    month = current_date.month
    quarter = f"Q{((month - 1) // 3) + 1}"

    # Telangana-friendly season logic
    if month in [6, 7, 8, 9]:
        season = "Monsoon"
    elif month in [10, 11, 12]:
        season = "Post-Monsoon"
    else:
        season = "Summer"

    date_rows.append([
        current_date.strftime("%Y-%m-%d"),
        year,
        month,
        quarter,
        season
    ])

dim_date = pd.DataFrame(
    date_rows,
    columns=["Date", "Year", "Month", "Quarter", "Season"]
)

# =========================================================
# HELPER LOOKUPS
# =========================================================
district_ids = dim_districts["District_ID"].tolist()
date_list = dim_date["Date"].tolist()

# population lookup
district_population = dict(zip(dim_districts["District_ID"], dim_districts["Population_2023"]))
district_industrial = dict(zip(dim_districts["District_ID"], dim_districts["Industrial_Zone_Count"]))
district_area = dict(zip(dim_districts["District_ID"], dim_districts["Total_Area_SQKM"]))

# season lookup
date_to_season = dict(zip(dim_date["Date"], dim_date["Season"]))
date_to_month = dict(zip(dim_date["Date"], dim_date["Month"]))

# =========================================================
# 3. FACT_RAINFALL TABLE
# =========================================================
# Exact row count for total = 20,000
fact_rows_each = 6534

rainfall_rows = []
for rec_id in range(1, fact_rows_each + 1):
    district_id = random.choice(district_ids)
    date = random.choice(date_list)
    season = date_to_season[date]
    month = date_to_month[date]

    # Historical average rainfall depends on month/season
    if season == "Monsoon":
        hist_avg = round(np.random.uniform(120, 280), 2)
        precipitation = max(0, round(np.random.normal(hist_avg, 45), 2))
    elif season == "Post-Monsoon":
        hist_avg = round(np.random.uniform(25, 100), 2)
        precipitation = max(0, round(np.random.normal(hist_avg, 20), 2))
    else:
        hist_avg = round(np.random.uniform(5, 40), 2)
        precipitation = max(0, round(np.random.normal(hist_avg, 12), 2))

    rainfall_rows.append([
        rec_id, district_id, date, precipitation, hist_avg
    ])

fact_rainfall = pd.DataFrame(
    rainfall_rows,
    columns=[
        "Record_ID", "District_ID", "Date",
        "Precipitation_mm", "Historical_Avg_mm"
    ]
)

# =========================================================
# 4. FACT_GROUNDWATER TABLE
# =========================================================
groundwater_rows = []
for meas_id in range(1, fact_rows_each + 1):
    district_id = random.choice(district_ids)
    date = random.choice(date_list)
    season = date_to_season[date]

    pop = district_population[district_id]
    industrial = district_industrial[district_id]

    # Urban growth + industrial pressure => deeper groundwater
    pressure_factor = (pop / 1000000) * 0.8 + industrial * 0.15

    if season == "Monsoon":
        depth = round(np.random.uniform(4, 14) + pressure_factor, 2)
        recharge = round(np.random.uniform(1.2, 4.8), 2)
    elif season == "Post-Monsoon":
        depth = round(np.random.uniform(6, 18) + pressure_factor, 2)
        recharge = round(np.random.uniform(0.8, 3.0), 2)
    else:
        depth = round(np.random.uniform(10, 24) + pressure_factor, 2)
        recharge = round(np.random.uniform(0.2, 1.8), 2)

    groundwater_rows.append([
        meas_id, district_id, date, depth, recharge
    ])

fact_groundwater = pd.DataFrame(
    groundwater_rows,
    columns=[
        "Measurement_ID", "District_ID", "Date",
        "Water_Table_Depth_m", "Recharge_Rate"
    ]
)

# =========================================================
# 5. FACT_RESERVOIRS TABLE
# =========================================================
reservoir_rows = []
for res_id in range(1, fact_rows_each + 1):
    district_id = random.choice(district_ids)
    date = random.choice(date_list)
    season = date_to_season[date]

    area = district_area[district_id]
    pop = district_population[district_id]

    # Bigger districts can have larger capacity
    max_capacity = round(np.random.uniform(3, 30) + (area / 1000) * 0.8, 2)

    # Population pressure reduces current fill stress
    demand_pressure = (pop / 1000000) * np.random.uniform(0.4, 1.1)

    if season == "Monsoon":
        current_capacity = round(np.random.uniform(0.60, 1.00) * max_capacity - demand_pressure, 2)
    elif season == "Post-Monsoon":
        current_capacity = round(np.random.uniform(0.40, 0.85) * max_capacity - demand_pressure, 2)
    else:
        current_capacity = round(np.random.uniform(0.15, 0.60) * max_capacity - demand_pressure, 2)

    current_capacity = max(0.1, min(current_capacity, max_capacity))

    reservoir_rows.append([
        res_id, district_id, date, current_capacity, max_capacity
    ])

fact_reservoirs = pd.DataFrame(
    reservoir_rows,
    columns=[
        "Res_ID", "District_ID", "Date",
        "Current_Capacity_TMC", "Max_Capacity_TMC"
    ]
)

# =========================================================
# CHECK TOTAL ROW COUNT
# =========================================================
total_rows = (
    len(dim_districts)
    + len(dim_date)
    + len(fact_rainfall)
    + len(fact_groundwater)
    + len(fact_reservoirs)
)

print("Total Rows Created:", total_rows)

# =========================================================
# SAVE TO SQLITE DATABASE
# =========================================================
conn = sqlite3.connect(db_name)

dim_districts.to_sql("Dim_Districts", conn, if_exists="replace", index=False)
dim_date.to_sql("Dim_Date", conn, if_exists="replace", index=False)
fact_rainfall.to_sql("Fact_Rainfall", conn, if_exists="replace", index=False)
fact_groundwater.to_sql("Fact_Groundwater", conn, if_exists="replace", index=False)
fact_reservoirs.to_sql("Fact_Reservoirs", conn, if_exists="replace", index=False)

# =========================================================
# OPTIONAL: ADD PRIMARY KEY / FOREIGN KEY STYLE SCHEMA
# SQLite via pandas doesn't enforce all constraints automatically,
# so we create clean SQL versions too.
# =========================================================
cur = conn.cursor()

# Drop temp tables and recreate with constraints
cur.executescript("""
DROP TABLE IF EXISTS Dim_Districts_Final;
DROP TABLE IF EXISTS Dim_Date_Final;
DROP TABLE IF EXISTS Fact_Rainfall_Final;
DROP TABLE IF EXISTS Fact_Groundwater_Final;
DROP TABLE IF EXISTS Fact_Reservoirs_Final;

CREATE TABLE Dim_Districts_Final (
    District_ID INTEGER PRIMARY KEY,
    Name TEXT,
    Total_Area_SQKM REAL,
    Population_2023 INTEGER,
    Industrial_Zone_Count INTEGER,
    Lat REAL,
    Long REAL
);

CREATE TABLE Dim_Date_Final (
    Date TEXT PRIMARY KEY,
    Year INTEGER,
    Month INTEGER,
    Quarter TEXT,
    Season TEXT
);

CREATE TABLE Fact_Rainfall_Final (
    Record_ID INTEGER PRIMARY KEY,
    District_ID INTEGER,
    Date TEXT,
    Precipitation_mm REAL,
    Historical_Avg_mm REAL,
    FOREIGN KEY (District_ID) REFERENCES Dim_Districts_Final(District_ID),
    FOREIGN KEY (Date) REFERENCES Dim_Date_Final(Date)
);

CREATE TABLE Fact_Groundwater_Final (
    Measurement_ID INTEGER PRIMARY KEY,
    District_ID INTEGER,
    Date TEXT,
    Water_Table_Depth_m REAL,
    Recharge_Rate REAL,
    FOREIGN KEY (District_ID) REFERENCES Dim_Districts_Final(District_ID),
    FOREIGN KEY (Date) REFERENCES Dim_Date_Final(Date)
);

CREATE TABLE Fact_Reservoirs_Final (
    Res_ID INTEGER PRIMARY KEY,
    District_ID INTEGER,
    Date TEXT,
    Current_Capacity_TMC REAL,
    Max_Capacity_TMC REAL,
    FOREIGN KEY (District_ID) REFERENCES Dim_Districts_Final(District_ID),
    FOREIGN KEY (Date) REFERENCES Dim_Date_Final(Date)
);
""")

dim_districts.to_sql("Dim_Districts_Final", conn, if_exists="append", index=False)
dim_date.to_sql("Dim_Date_Final", conn, if_exists="append", index=False)
fact_rainfall.to_sql("Fact_Rainfall_Final", conn, if_exists="append", index=False)
fact_groundwater.to_sql("Fact_Groundwater_Final", conn, if_exists="append", index=False)
fact_reservoirs.to_sql("Fact_Reservoirs_Final", conn, if_exists="append", index=False)

conn.commit()
conn.close()

# =========================================================
# EXPORT CSV FILES
# =========================================================
dim_districts.to_csv("Dim_Districts.csv", index=False)
dim_date.to_csv("Dim_Date.csv", index=False)
fact_rainfall.to_csv("Fact_Rainfall.csv", index=False)
fact_groundwater.to_csv("Fact_Groundwater.csv", index=False)
fact_reservoirs.to_csv("Fact_Reservoirs.csv", index=False)

# =========================================================
# PREVIEW
# =========================================================
print("\nDim_Districts:")
print(dim_districts.head())

print("\nDim_Date:")
print(dim_date.head())

print("\nFact_Rainfall:")
print(fact_rainfall.head())

print("\nFact_Groundwater:")
print(fact_groundwater.head())

print("\nFact_Reservoirs:")
print(fact_reservoirs.head())

print("\nFiles created successfully:")
print("- telangana_water_stress_star_schema.db")
print("- Dim_Districts.csv")
print("- Dim_Date.csv")
print("- Fact_Rainfall.csv")
print("- Fact_Groundwater.csv")
print("- Fact_Reservoirs.csv")

Total Rows Created: 20000

Dim_Districts:
   District_ID                  Name  Total_Area_SQKM  Population_2023  \
0            1              Adilabad          2811.26          4293000   
1            2  Bhadradri Kothagudem          3003.75           719894   
2            3            Hanamkonda          3711.80           386454   
3            4             Hyderabad          1802.10          4467288   
4            5               Jagtial          2966.25          1523162   

   Industrial_Zone_Count        Lat       Long  
0                     11  18.996733  79.468031  
1                     11  19.351322  79.484237  
2                      2  18.760195  80.766500  
3                      1  17.047393  79.194074  
4                     11  17.439430  77.377330  

Dim_Date:
         Date  Year  Month Quarter  Season
0  2023-01-01  2023      1      Q1  Summer
1  2023-01-02  2023      1      Q1  Summer
2  2023-01-03  2023      1      Q1  Summer
3  2023-01-04  2023      1      Q1  